In [ ]:
!pip install langchain==0.3.27
!pip install langchain-aws==0.2.34
!pip install langchain-community==0.3.31
!pip install langchain-core==0.3.79
!pip install langchain-experimental==0.3.4

### AWS Bedrock LLM Setup

In [13]:
# AWS Bedrock LLM Setup
# ---------------------------
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings
#llm = ChatBedrockConverse(model_id="amazon.nova-lite-v1:0", region_name="us-east-1", temperature=0.5, max_tokens=50)
llm = ChatBedrockConverse(model_id="cohere.command-r-plus-v1:0", region_name="us-east-1", temperature=0.5, max_tokens=500)


In [8]:
llm.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '4b0ea823-84f2-4a6e-9949-37255fe63fad', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Wed, 22 Oct 2025 05:42:44 GMT', 'content-type': 'application/json', 'content-length': '232', 'connection': 'keep-alive', 'x-amzn-requestid': '4b0ea823-84f2-4a6e-9949-37255fe63fad'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [375]}, 'model_name': 'cohere.command-r-plus-v1:0'}, id='run--7ee88359-7bad-46e7-84ba-4c908e151097-0', usage_metadata={'input_tokens': 1, 'output_tokens': 9, 'total_tokens': 10, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})

## Load the file

In [14]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("bns_v2.pdf")
documents = loader.load()
documents


[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20251020151201', 'source': 'bns_v2.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1'}, page_content='THE BHARA TIYA NY AYA SANHITA, 2023\nNO. 45 OF 2023\n[25th December ,2023.]\nAn Act to consolidate and amend the provisions relating to offences and for\nmatters connected therewithor incidental thereto.\nBE it enacted by Parliament in the Seventy-fourth Year of the Republic of India as\nfollows:––\nCHAPTERI\nPRELIMINARY\n1.(1) This Act may be called the Bharatiya Nyaya Sanhita, 2023.\n(2) It shall come into force on such date as the Central Government may , bynotification\nin the Official Gazette, appoint, and different dates maybe appointed for different provisions\nof this Sanhita.\nShort title,\ncommencement\nand\napplication.\nvlk/kkj.k\nEXTRAORDINARY\nHkkx II — [k.M 1\nPART II — Section 1\nizkf/kdkj ls izdkf\'kr\nPUBLISHED BY AUTHORITY\nlañ 53] ubZ fnYyh] lkseokj] fnlEcj 25] 2023@ ikS"k 4] 1945 ¼

## Perform Chunking

In [15]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n","\n", ""],
    chunk_size = 1000,
    chunk_overlap = 200,
    #length_function = len_func,
    is_separator_regex=True
)
chunk_list = text_splitter.split_documents(documents)
chunk_list[0]


Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20251020151201', 'source': 'bns_v2.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1'}, page_content='THE BHARA TIYA NY AYA SANHITA, 2023\nNO. 45 OF 2023\n[25th December ,2023.]\nAn Act to consolidate and amend the provisions relating to offences and for\nmatters connected therewithor incidental thereto.\nBE it enacted by Parliament in the Seventy-fourth Year of the Republic of India as\nfollows:––\nCHAPTERI\nPRELIMINARY\n1.(1) This Act may be called the Bharatiya Nyaya Sanhita, 2023.\n(2) It shall come into force on such date as the Central Government may , bynotification\nin the Official Gazette, appoint, and different dates maybe appointed for different provisions\nof this Sanhita.\nShort title,\ncommencement\nand\napplication.\nvlk/kkj.k\nEXTRAORDINARY\nHkkx II — [k.M 1\nPART II — Section 1\nizkf/kdkj ls izdkf\'kr\nPUBLISHED BY AUTHORITY\nlañ 53] ubZ fnYyh] lkseokj] fnlEcj 25] 2023@ ikS"k 4] 1945 ¼\

## Create Embedding Model Instance

In [16]:
from langchain_aws import BedrockEmbeddings
embedding_model = BedrockEmbeddings(model_id="amazon.titan-embed-text-v1")
text = "Example text for understanding text-embeddings"
query_result = embedding_model.embed_query(text)
query_result[:5]


[0.11181640625, 0.0771484375, -0.37109375, 0.53125, -0.10400390625]

### Create Vector Store - Chroma

In [17]:
# Import Chroma vector store from LangChain
from langchain.vectorstores import Chroma
# Import ChromaDB configuration (optional, useful for advanced setup or customization)
import chromadb.config
# Create the vector store using Chroma from a list of documents and their embeddings
# This will serve as the index for similarity search and retrieval
db = Chroma.from_documents(chunk_list, 
                           embedding=embedding_model,
                           collection_name='bns2',
                           persist_directory='./bns2')


In [18]:
db.similarity_search_with_score("Act done by a person bound,or by mistakeof fact believing himself bound,by law",k=3)

[(Document(metadata={'page_label': '11', 'page': 10, 'creator': 'PDFium', 'source': 'bns_v2.pdf', 'producer': 'PDFium', 'total_pages': 20, 'creationdate': 'D:20251020151201'}, page_content='by reason of a mistake of fact and not by reason of a mistake of law in good faith, believes\nhimself to be justified by law, in doing it.\nIllustration.\nA sees Z commit what appears to A to be a murder. A, in the exercise, to the best of his\njudgment exerted in good faith, of the power which the law gives to all persons of apprehending\nmurderers in the fact, seizes Z, in order to bring Z before the proper authorities. A has\ncommitted no offence, though it may turn out that Z was acting in self-defence.\n18.Nothing is an offence which is done by accident or misfortune, and without any\ncriminal intention or knowledge in the doing of a lawful act in a lawful manner by lawful\nmeans and with proper care and caution.\nIllustration.\nA is at work with a hatchet; the head flies off and kills a man wh

## Create Prompt Template 

In [19]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain.prompts import ChatPromptTemplate

system_template = (
    "You are an laws-assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "\n\nContext: {context}"
)

# Create the ChatPromptTemplate
# retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")
retrieval_qa_chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_template),
        ("human", "{input}"),
    ]
)

### Use VDB as a retriever and perform retrieval augmented generation

In [20]:

retriever = db.as_retriever()
combine_docs_chain = create_stuff_documents_chain(llm, retrieval_qa_chat_prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

response=retrieval_chain.invoke({"input": "what is meant by act done by a person bound,or by mistakeof fact believing himself bound,by law?"})

In [21]:
response

{'input': 'what is meant by act done by a person bound,or by mistakeof fact believing himself bound,by law?',
 'context': [Document(metadata={'creationdate': 'D:20251020151201', 'total_pages': 20, 'producer': 'PDFium', 'creator': 'PDFium', 'page': 10, 'source': 'bns_v2.pdf', 'page_label': '11'}, page_content='by reason of a mistake of fact and not by reason of a mistake of law in good faith, believes\nhimself to be justified by law, in doing it.\nIllustration.\nA sees Z commit what appears to A to be a murder. A, in the exercise, to the best of his\njudgment exerted in good faith, of the power which the law gives to all persons of apprehending\nmurderers in the fact, seizes Z, in order to bring Z before the proper authorities. A has\ncommitted no offence, though it may turn out that Z was acting in self-defence.\n18.Nothing is an offence which is done by accident or misfortune, and without any\ncriminal intention or knowledge in the doing of a lawful act in a lawful manner by lawful\nm

In [22]:
response['answer']

'This refers to a situation where a person, due to a mistake of fact and not a mistake of law, believes in good faith that they are justified by law to perform a certain act. In simpler terms, it means that the person mistakenly believes that they are legally obligated or permitted to do something, based on their understanding of the facts, even though their belief may be incorrect.'

# Rag With Ragas Evaluation Framework

### Create sample Q and A

In [ ]:
#!pip install ragas

In [23]:
sample_queries = [
    "What does the term 'illegal' apply to, as per the Sanhita?",
    "What are the four main types of punishments specified in Section 4 (Chapter II)?",
]

expected_responses = [
    "The word \"illegal\" is applicable to everything which is an offence or which is prohibited by law, or which furnishes ground for a civil action.",
    "The main punishments specified are Death, Imprisonment for life, Imprisonment (Rigorous or Simple), and Fine (Community Service is added by amendment/context).",
]

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


template = """Answer the question based only on the following context:
{context}

Question: {query}
"""
prompt = ChatPromptTemplate.from_template(template)

qa_chain = prompt | llm | StrOutputParser()

In [27]:
retriever = db.as_retriever()

In [30]:
def format_docs(relevant_docs):
    return "\n".join(doc.page_content for doc in relevant_docs)


# query = "Your query here"

# relevant_docs = retriever.invoke(query)
# qa_chain.invoke({"context": format_docs(relevant_docs), "query": query})

### Create evaluation Dataset

In [31]:
from ragas import EvaluationDataset


dataset = []

for query, reference in zip(sample_queries, expected_responses):
    relevant_docs = retriever.invoke(query)
    response = qa_chain.invoke({"context": format_docs(relevant_docs), "query": query})
    dataset.append(
        {
            "user_input": query,
            "retrieved_contexts": [rdoc.page_content for rdoc in relevant_docs],
            "response": response,
            "reference": reference,
        }
    )

evaluation_dataset = EvaluationDataset.from_list(dataset)

In [38]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]━━ 1/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [39]:
import pandas as pd
df=evaluation_dataset.to_pandas()
df.to_excel('eval.xlsx')

### Instantiate the Bedrock LLM for Evaluation

In [41]:
# Required Imports
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
# Use the class names for metrics (e.g., LLMContextRecall, not context_recall)
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness 

# Correct LangChain Bedrock Import
from langchain_aws.chat_models import ChatBedrockConverse
from langchain_core.language_models.chat_models import BaseChatModel 

# --- 1. Instantiate the Bedrock LLM for Evaluation ---
# Use a robust model (like Command R-Plus or Claude 3 Sonnet) for the evaluator 
# to improve JSON parsing success.

evaluator_llm_instance: BaseChatModel = ChatBedrockConverse(
    model_id="cohere.command-r-plus-v1:0", 
    region_name="us-east-1", # Set your AWS region
    temperature=0, 
    max_tokens=2048 # Recommended to avoid truncation of Ragas's JSON output
)

### Create the Ragas LLM Wrapper and Run the Evaluation 

In [42]:
evaluator_llm = LangchainLLMWrapper(evaluator_llm_instance) 

result = evaluate(
    dataset=evaluation_dataset,
    # Pass the instantiated metric classes (use the FactualCorrectness class, not the renamed function)
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
    # Crucial for stability: Skip samples that fail due to parsing/timeout
    raise_exceptions=False, 
)

print(result)

/tmp/ipykernel_1768/1104488675.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  evaluator_llm = LangchainLLMWrapper(evaluator_llm_instance)


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

Exception raised in Job[5]: TimeoutError()


{'context_recall': 1.0000, 'faithfulness': 0.6000, 'factual_correctness(mode=f1)': 0.5000}
